# Extra check 3 - Direct `<unk>`: how sure is the model when it writes `<unk>`?

At the step where the model starts to write `<unk>`, how sure is it, and what would it have written instead? A high probability would look like a learned habit; a low one, with other candidates close behind, would look like a fallback for uncertainty. The result (middle values, with the runner-up sometimes close) fits both readings, so this check does not tell them apart.

Text-only input (extra check 1 showed that text input reproduces the `<unk>`), same model and pinned revision, greedy decoding, one sentence at a time, FP16. For every output step we save the chosen token, its probability, the entropy and the top-5 candidates, for two groups of unique sentences: those whose original speech output had a visible `<unk>`, and the same number of sentences without `<unk>`, matched on sentence length (seed 760). It is a post-hoc check; the original 4,200 predictions are not changed. The statistics printed at the end of this notebook are a first look (all other steps in the original `unk` group, token-level Mann-Whitney test). The final comparison, which uses the other Chinese-character steps in the sentences that wrote `<unk>` and a sentence-level bootstrap, is in `scripts/decode_summary.py`.

Input: `merged_3system_with_metrics.csv` in your Google Drive folder (default `MyDrive/cs760`). Results are saved to a subfolder of it every 50 sentences, so a disconnected run can be resumed; the GPU is released at the end.

In [ ]:
%pip install -q "transformers>=4.40,<5" sentencepiece protobuf scipy pandas tqdm

In [ ]:
import os, json, re, time
from pathlib import Path

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/cs760"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path(DRIVE_DIR)
else:
    DATA_DIR = Path(".")

import numpy as np, pandas as pd, torch, transformers
from tqdm.auto import tqdm
from transformers import AutoProcessor, SeamlessM4Tv2ForTextToText

MODEL_ID = "facebook/seamless-m4t-v2-large"
MODEL_REVISION = "5f8cc790b19fc3f67a61c105133b20b34e3dcb76"   # use the same one as the main Direct run
TARGET_LANGUAGE, SOURCE_LANGUAGE = "cmn", "eng"
GENERATION_KWARGS = {"num_beams": 1, "do_sample": False, "max_new_tokens": 256}   # keep the same as main run

MERGED_CSV = DATA_DIR / "merged_3system_with_metrics.csv"
OUT_DIR = DATA_DIR / "decode_probs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 760
MAX_MINUTES = 20
AUTO_RELEASE_RUNTIME = True    # only if USE_DRIVE is True
# True = only 6 sentences (3 + 3)
SMOKE = False 
# Smoke runs write to their own files (sentence ids differ between the two)
SUFFIX = "_SMOKE" if SMOKE else ""
STEPS_CSV, SENT_CSV = OUT_DIR / f"decode_steps{SUFFIX}.csv", OUT_DIR / f"decode_sentences{SUFFIX}.csv"

CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA else "cpu")
DTYPE = torch.float16 if CUDA else torch.float32
print("device:", DEVICE, "| dtype:", DTYPE, "| transformers:", transformers.__version__)
if not CUDA:
    print("WARNING: no GPU. Slow, and not identical to the T4/FP16 run.")

In [ ]:
from huggingface_hub import login
login()   # use your own Hugging Face token

In [ ]:
df = pd.read_csv(MERGED_CSV)
assert len(df) == 4200
df["words"] = df.sentence.str.split().str.len()
# one row per unique sentence; 'unk' if ANY clip of that sentence had a visible <unk> in the original speech output
u = df.groupby("sentence").agg(speech_unk=("direct_has_unk", "max"), words=("words", "first"), n_clips=("id", "size")).reset_index()
u["bin"] = pd.cut(u.words, [0, 5, 8, 11, 15, 1000], labels=["1-5", "6-8", "9-11", "12-15", "16+"])
unk = u[u.speech_unk]
rng = np.random.default_rng(SEED)
ctrl_parts = []
for b, g in unk.groupby("bin", observed=True):
    pool = u[(~u.speech_unk) & (u.bin == b)]
    ctrl_parts.append(pool.sample(n=min(len(g), len(pool)), random_state=int(rng.integers(1e9))))
ctrl = pd.concat(ctrl_parts)
unk = unk.assign(group="unk"); ctrl = ctrl.assign(group="control")
if SMOKE:
    unk, ctrl = unk.head(3), ctrl.head(3)
todo = pd.concat([unk, ctrl]).sample(frac=1, random_state=SEED).reset_index(drop=True)   # shuffled: any prefix is a mix
todo["sid"] = np.arange(len(todo))
print(f"unique sentences: unk group {len(unk)}, control group {len(ctrl)} (length-matched)")
print(todo.groupby(["group", "bin"], observed=True).size().unstack(0))

# resume
steps_rows = pd.read_csv(STEPS_CSV).to_dict("records") if STEPS_CSV.exists() else []
sent_rows = pd.read_csv(SENT_CSV).to_dict("records") if SENT_CSV.exists() else []
done = {r["sentence"] for r in sent_rows}
todo_left = todo[~todo.sentence.isin(done)]
print("already done:", len(done), "| still to run:", len(todo_left))

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = SeamlessM4Tv2ForTextToText.from_pretrained(MODEL_ID, revision=MODEL_REVISION, dtype=DTYPE)
model.to(DEVICE).eval()
tok = processor.tokenizer
UNK_ID = tok.unk_token_id
# the spelled-out marker: '<' or '▁<'  +  'unk'  +  '>'   (token ids seen in the team's log)
START_IDS, MID_ID = {249371, 9614}, 2105
print("unk_token_id:", UNK_ID, "| marker start ids:", START_IDS, "| middle id:", MID_ID)

In [ ]:
@torch.inference_mode()
def run_sentence(text, sid):
    enc = processor(text=[text], src_lang=SOURCE_LANGUAGE, return_tensors="pt").to(DEVICE)
    gen = model.generate(**enc, tgt_lang=TARGET_LANGUAGE, **GENERATION_KWARGS,
                         return_dict_in_generate=True, output_logits=True)
    seq = gen.sequences[0].tolist()
    logits = torch.stack(gen.logits, dim=0)[:, 0, :].float()          # (steps, vocab): raw logits per generated step
    offset = len(seq) - logits.shape[0]                                # tokens given at the start (decoder start, language tag)
    assert 0 <= offset <= 3, f"unexpected alignment: len(seq)={len(seq)}, steps={logits.shape[0]}"
    probs = torch.softmax(logits, dim=-1)
    ent = -(probs * torch.log(probs.clamp_min(1e-12))).sum(-1)
    top_p, top_i = probs.topk(5, dim=-1)
    start_mass = probs[:, list(START_IDS)].sum(-1)                     # probability mass on '<' / '▁<' at every step
    ids = seq[offset:]
    chosen = torch.tensor(ids, device=probs.device)
    p_ch = probs.gather(1, chosen[:, None])[:, 0].tolist()            # one GPU->CPU copy per tensor, not one per number
    ent, start_mass = ent.tolist(), start_mass.tolist()
    top_p, top_i = top_p.tolist(), top_i.tolist()
    rows = []
    for t in range(len(ids)):
        is_start = (ids[t] in START_IDS and t + 1 < len(ids) and ids[t + 1] == MID_ID)
        rows.append(dict(sid=sid, step=t, tok_id=ids[t], tok=tok.convert_ids_to_tokens(ids[t]),
                         p_chosen=p_ch[t], entropy=ent[t], unk_start_mass=start_mass[t], is_unk_start=is_start,
                         **{f"alt{k+1}": tok.convert_ids_to_tokens(top_i[t][k]) for k in range(5)},
                         **{f"p{k+1}": top_p[t][k] for k in range(5)}))
    text_out = processor.decode(seq, skip_special_tokens=True).strip()
    return rows, text_out, len(ids)

if len(todo_left):
    _ = run_sentence(todo_left.sentence.iloc[0], -1)       # warm-up (not saved)

def save():
    pd.DataFrame(steps_rows).to_csv(STEPS_CSV, index=False)
    pd.DataFrame(sent_rows).to_csv(SENT_CSV, index=False)

t0 = time.time(); stopped_by = "finished all sentences"
for k, r in enumerate(tqdm(todo_left.itertuples(), total=len(todo_left)), start=1):
    rows, out, n = run_sentence(r.sentence, int(r.sid))
    steps_rows += rows
    sent_rows.append(dict(sid=int(r.sid), group=r.group, sentence=r.sentence, words=int(r.words), n_steps=n, translation=out))
    if k % 50 == 0:
        save()
    if (time.time() - t0) / 60 > MAX_MINUTES:
        stopped_by = f"time budget reached ({MAX_MINUTES} min)"; break
save()
steps = pd.DataFrame(steps_rows); sents = pd.DataFrame(sent_rows)
print(f"sentences: {len(sents)} | steps: {len(steps)} | stopped by: {stopped_by} | this session: {(time.time()-t0)/60:.1f} min")

In [ ]:
# Free the GPU memory now (the analysis below needs no GPU)
del model
import gc; gc.collect()
if CUDA:
    torch.cuda.empty_cache()

In [ ]:
from scipy.stats import mannwhitneyu
UNK_RE = re.compile(r"<\s*unk\s*>", re.IGNORECASE)
sents["text_has_unk"] = sents.translation.fillna("").map(lambda s: bool(UNK_RE.search(s)))
steps = steps.merge(sents[["sid", "group", "text_has_unk"]], on="sid")
steps["is_unk_start"] = steps.is_unk_start.astype(bool)
steps["is_han"] = steps.tok.astype(str).str.contains(r"[\u4e00-\u9fff]")

print("unk group sentences that produced <unk> in this run:", int(sents[sents.group == 'unk'].text_has_unk.sum()), "of", int((sents.group == 'unk').sum()))
print("control group sentences that produced <unk>:", int(sents[sents.group == 'control'].text_has_unk.sum()), "of", int((sents.group == 'control').sum()))

U = steps[steps.is_unk_start]
others_unk = steps[(steps.group == "unk") & ~steps.is_unk_start]
ctrl_steps = steps[steps.group == "control"]
def desc(name, d):
    return dict(steps=len(d), p_chosen_median=d.p_chosen.median(), p_chosen_mean=d.p_chosen.mean(),
                entropy_median=d.entropy.median(), share_p_over_0_5=(d.p_chosen > 0.5).mean(), share_p_over_0_9=(d.p_chosen > 0.9).mean(), group=name)
cmp_ = pd.DataFrame([desc("A. steps where <unk> starts", U),
                     desc("B. all other steps, same sentences", others_unk),
                     desc("C. steps in control sentences", ctrl_steps),
                     desc("D. Chinese-character steps, control", ctrl_steps[ctrl_steps.is_han])]).set_index("group").round(3)
print(); print(cmp_.to_string())
if len(U) > 5:
    print("\nMann-Whitney, p_chosen at <unk> start vs other steps (same sentences): p =", mannwhitneyu(U.p_chosen, others_unk.p_chosen).pvalue)

In [ ]:
# How much probability does the model put on starting '<unk>' at ANY step? (even when it does not write it)
mx = steps.groupby(["sid", "group"]).unk_start_mass.max().reset_index()
print(mx.groupby("group").unk_start_mass.describe().round(3).to_string())
print("\nshare of sentences where the model puts more than 5% / 20% probability on '<' at some step:")
print(mx.assign(over5=mx.unk_start_mass > 0.05, over20=mx.unk_start_mass > 0.20).groupby("group")[["over5", "over20"]].mean().round(3).to_string())

In [ ]:
# Look at it step by step: what did the model write, and what was next in line?
ex = U.merge(sents[["sid", "sentence", "translation"]], on="sid").sort_values("p_chosen")
def show(r):
    alts = ", ".join(f"{r[f'alt{k}']} ({r[f'p{k}']:.2f})" for k in range(1, 6))
    print(f"EN : {r.sentence}\nOUT: {r.translation}\nAt the '<unk>' step: chose {r.tok!r} with p={r.p_chosen:.2f}, entropy={r.entropy:.2f}\nTop 5: {alts}\n")
print("=== least sure (lowest probability) ===")
for _, r in ex.head(8).iterrows(): show(r)
print("=== most sure (highest probability) ===")
for _, r in ex.tail(8).iterrows(): show(r)

In [ ]:
# What did the model rank first after '<'-type tokens? Count the most common alternatives at '<unk>' steps (excluding the '<' variants)
alt_counts = {}
for _, r in U.iterrows():
    for k in range(1, 6):
        a = r[f"alt{k}"]
        if a not in ("<", "▁<"):
            alt_counts[a] = alt_counts.get(a, 0) + 1
print(pd.Series(alt_counts).sort_values(ascending=False).head(25).to_string())

In [ ]:
summary = {
    "model_id": MODEL_ID, "revision": MODEL_REVISION, "device": str(DEVICE), "dtype": str(DTYPE),
    "generation": GENERATION_KWARGS, "seed": SEED, "stopped_by": stopped_by,
    "sentences": int(len(sents)), "unk_group": int((sents.group == "unk").sum()), "control_group": int((sents.group == "control").sum()),
    "unk_start_steps": int(len(U)),
    "unk_start_p_chosen_median": float(U.p_chosen.median()) if len(U) else None,
    "other_steps_p_chosen_median": float(others_unk.p_chosen.median()),
    "unk_start_entropy_median": float(U.entropy.median()) if len(U) else None,
    "other_steps_entropy_median": float(others_unk.entropy.median()),
    "note": "Post-hoc diagnostic (text input). Original 4,200 Direct predictions unchanged.",
}
(OUT_DIR / f"run_summary_unk_extra_3_decode_probs{SUFFIX}.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2)); print("\nResults saved to:", OUT_DIR)

In [ ]:
# release the GPU; the results are already on Google Drive
if USE_DRIVE and AUTO_RELEASE_RUNTIME:
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print("Could not release automatically:", e, "\nUse Runtime -> Disconnect and delete runtime.")
else:
    print("Remember: Runtime -> Disconnect and delete runtime, to stop using GPU quota.")